In [ ]:
!pip install naiveautoml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 1.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Si estás en Colab, descomenta estas líneas para subir el archivo:
from google.colab import files
uploaded = files.upload()

Saving Dataset_ML_Clasificacion_SLA_4.csv to Dataset_ML_Clasificacion_SLA_4.csv


In [ ]:
import logging
import pandas as pd
import matplotlib.pyplot as plt
import naiveautoml
from sklearn.model_selection import train_test_split

# =============================================================================
# 1. CONFIGURACIÓN DE OBSERVABILIDAD (LOGGING)
# =============================================================================
logger = logging.getLogger('naiveautoml')
logger.setLevel(logging.INFO)
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
ch.setFormatter(formatter)
logger.addHandler(ch)

logger_eval = logging.getLogger('naiveautoml.evalpool')
logger_eval.setLevel(logging.WARN)
logger_eval.addHandler(ch)

# =============================================================================
# 2. CARGA DE DATOS Y PREVENCIÓN DE FUGA / OVERFITTING (CRÍTICO)
# =============================================================================
print("Cargando el Dataset Consolidado...")
# Ajustado al nombre real de tu dataset depurado original (Sin tocar por Excel)
df = pd.read_csv('Dataset_ML_Clasificacion_SLA_4.csv')

# --- PURGA ESTRATÉGICA (Drop) ---
# Eliminamos todo lo que cause ruido, overfitting o data leakage (trampa)
columnas_a_eliminar = [
    # A. Metadatos e IDs (Alta cardinalidad = El modelo los memoriza y no generaliza)
    'Grupo_Sesion_ID', 'JobID', 'BackupID', 'Cadena_ID',

    # B. Formato Datetime (Los modelos matemáticos no procesan fechas crudas)
    'Start_Time', 'Expire_Time',

    # C. PREVENCIÓN DE FUGA DE INFORMACIÓN Y COLINEALIDAD
    'SLA_RPO_Cumple', 'SLA_RTO_Cumple', 'Cumple_SLA', # Targets intermedios y final
    'RPO_Horas', 'RTO_Estimado_Horas',                # Padres matemáticos del target
    'Umbral_RPO_Horas', 'Umbral_RTO_Horas',           # Constantes (Varianza Cero)
    'ElapsedSeconds',                                 # Alta colinealidad con Duracion_Sesion_Segundos

    # D. Consecuencias (No se conocen hasta DESPUÉS de evaluar el SLA)
    'Riesgo_Total_COP'
]

# Separamos predictoras (X) de objetivo (y)
X_cruda = df.drop(columns=columnas_a_eliminar, errors='ignore')
y = df['Cumple_SLA']

# =============================================================================
# 3. TRANSFORMACIÓN ONE-HOT ENCODING (Variables de Texto)
# =============================================================================
# Garantiza que todos los algoritmos evalúen sin errores de "Not a Number (NaN)"
columnas_texto = ['Nodo', 'Aplicacion', 'ScheduleLabel']
X = pd.get_dummies(X_cruda, columns=columnas_texto, drop_first=True)

# Convertimos booleanos a flotantes para compatibilidad total con SciKit-Learn
X = X.astype(float)

# Separación 80/20 garantizando reproducibilidad académica
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dimensiones de Entrenamiento (X_train) post-Encoding: {X_train.shape}")
print(f"Variables Matemáticas (Features) que el modelo evaluará:\n{X_train.columns.tolist()}")


# =============================================================================
# 4. INICIALIZACIÓN DE NAIVE AUTOML (Tus parámetros exitosos)
# =============================================================================
naml = naiveautoml.NaiveAutoML(
    show_progress=True,
    timeout_overall=360,          # Configuración validada en tu entorno
    max_hpo_iterations=100
)

# =============================================================================
# 5. ENTRENAMIENTO DEL MODELO (FIT)
# =============================================================================
# Pasamos las variables categóricas representadas por NÚMEROS
# Para que el algoritmo sepa que "Dia 6" no vale 6 veces más que "Dia 1".
variables_categoricas_numericas = ['Hora_Dia', 'Dia_Semana', 'Es_Fin_Semana', 'Inicio_Cadena']

print("\nIniciando búsqueda automatizada del mejor modelo (AutoML)...")
naml.fit(X_train, y_train, categorical_features=variables_categoricas_numericas)


# =============================================================================
# 6. RESULTADOS Y EVALUACIÓN
# =============================================================================
print("\n" + "="*50)
print("--- MEJOR MODELO ENCONTRADO POR LA BÚSQUEDA ---")
print("="*50)
print(naml.chosen_model)

print("\n--- TABLA DE CLASIFICACIÓN (LEADERBOARD) ---")
print(naml.leaderboard)

# =============================================================================
# 7. VISUALIZACIÓN DEL HISTORIAL DE APRENDIZAJE
# =============================================================================
try:
    history = naml.history
    scores = [h['score'] for h in history]
    tiempos = [h['time'] for h in history]

    plt.figure(figsize=(10, 5))
    plt.plot(tiempos, scores, marker='o', linestyle='-', color='b')
    plt.title('Evolución de la precisión durante la búsqueda de NaiveAutoML')
    plt.xlabel('Tiempo (Segundos)')
    plt.ylabel('Score (Precisión Interna)')
    plt.grid(True)
    plt.show()
except Exception as e:
    print(f"\nNo se pudo generar la gráfica de historial: {e}")

2026-04-22 15:42:41,864 - naiveautoml - INFO - Automatically inferred task type: classification
2026-04-22 15:42:41,864 - naiveautoml - INFO - Automatically inferred task type: classification
INFO:naiveautoml:Automatically inferred task type: classification
2026-04-22 15:42:41,892 - naiveautoml - INFO - There are 4 categorical features, which will be binarized.
2026-04-22 15:42:41,892 - naiveautoml - INFO - There are 4 categorical features, which will be binarized.
INFO:naiveautoml:There are 4 categorical features, which will be binarized.
2026-04-22 15:42:41,894 - naiveautoml - INFO - Missing values for the different attributes are RetDays                      0
Hora_Dia                     0
Dia_Semana                   0
Es_Fin_Semana                0
Volumen_Total_MB             0
Num_Hilos                    0
Duracion_Sesion_Segundos     0
Efecto_Canales               0
Velocidad_MB_s               0
Inicio_Cadena                0
Volumen_Acumulado_GB         0
Nodo_Nodo2        

Cargando el Dataset Consolidado...
Dimensiones de Entrenamiento (X_train) post-Encoding: (2753, 20)
Variables Matemáticas (Features) que el modelo evaluará:
['RetDays', 'Hora_Dia', 'Dia_Semana', 'Es_Fin_Semana', 'Volumen_Total_MB', 'Num_Hilos', 'Duracion_Sesion_Segundos', 'Efecto_Canales', 'Velocidad_MB_s', 'Inicio_Cadena', 'Volumen_Acumulado_GB', 'Nodo_Nodo2', 'Aplicacion_APP12', 'Aplicacion_APP3', 'Aplicacion_APP4', 'Aplicacion_APP5', 'Aplicacion_APP9', 'ScheduleLabel_CUMULATIVO', 'ScheduleLabel_DIFERENCIAL', 'ScheduleLabel_FULL']

Iniciando búsqueda automatizada del mejor modelo (AutoML)...
Progress for algorithm selection:


  0%|          | 0/32 [00:00<?, ?it/s]2026-04-22 15:42:41,910 - naiveautoml - INFO - --------------------------------------------------
2026-04-22 15:42:41,910 - naiveautoml - INFO - --------------------------------------------------
INFO:naiveautoml:--------------------------------------------------
2026-04-22 15:42:41,913 - naiveautoml - INFO - Selecting component for step with name: learner
2026-04-22 15:42:41,913 - naiveautoml - INFO - Selecting component for step with name: learner
INFO:naiveautoml:Selecting component for step with name: learner
2026-04-22 15:42:41,918 - naiveautoml - INFO - --------------------------------------------------
2026-04-22 15:42:41,918 - naiveautoml - INFO - --------------------------------------------------
INFO:naiveautoml:--------------------------------------------------
2026-04-22 15:42:41,920 - naiveautoml - INFO - Evaluating sklearn.ensemble._forest.ExtraTreesClassifier.Timeout: 300. Remaining time: 349.94328260421753
2026-04-22 15:42:41,920 - 

Progress for hyperparameter optimization:


  0%|          | 0/100 [00:00<?, ?it/s]2026-04-22 15:48:03,714 - naiveautoml - INFO - --------------------------------------------------
2026-04-22 15:48:03,714 - naiveautoml - INFO - --------------------------------------------------
INFO:naiveautoml:--------------------------------------------------
2026-04-22 15:48:03,716 - naiveautoml - INFO - Entering HPO phase.Remaining time: 38.15s
2026-04-22 15:48:03,716 - naiveautoml - INFO - Entering HPO phase.Remaining time: 38.15s
INFO:naiveautoml:Entering HPO phase.Remaining time: 38.15s
2026-04-22 15:48:03,722 - naiveautoml - INFO - --------------------------------------------------
2026-04-22 15:48:03,722 - naiveautoml - INFO - --------------------------------------------------
INFO:naiveautoml:--------------------------------------------------
2026-04-22 15:48:03,723 - naiveautoml - INFO - Starting 1-th HPO step. Currently best known score is -inf
2026-04-22 15:48:03,723 - naiveautoml - INFO - Starting 1-th HPO step. Currently best know


--- MEJOR MODELO ENCONTRADO POR LA BÚSQUEDA ---
Pipeline(steps=[('impute_and_binarize',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['RetDays',
                                                   'Volumen_Total_MB',
                                                   'Num_Hilos',
                                                   'Duracion_Sesion_Segundos',
                                                   'Efecto_Canales',
                                                   'Velocidad_MB_s',
                                                   'Volumen_Acumulado_GB',
                                                   'Nodo_Nodo2',
                                                   'Aplicacion_APP12',
                                               